# Sample covariance and shrinkage covariance

This notebook compares the sample covariance with `shrink_cov`. Each day it takes the previous month's CSI 300 weights, keeps the 100 largest names with a complete prior 60 trading days, and forms a long-only global minimum-variance portfolio. Two results are recorded: the condition number of the covariance, and the realized volatility over the following 60 days.


# 样本协方差和收缩协方差

比较样本协方差和 `shrink_cov`。每一天用上一个月的沪深 300 权重，取过去 60 个交易日数据完整、权重最大的 100 只，做只做多的全局最小方差。看两件事：协方差矩阵的条件数，以及之后 60 天的已实现波动。


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import edhec_risk_kit as erk


## Shrinkage formula

`shrink_cov` starts from the sample covariance S divided by T. The target F is the average variance times the identity. `γ = ||S − F||²`, and `π` sums the estimated sampling variance of every covariance entry. The intensity is `δ = min(π / (γ T), 1)`, and the estimate is `(1 − δ) S + δ F`. This is a simplified Ledoit-Wolf intensity, without the paper's ρ correction for error in the target. `shrinkage_cov` is a different helper: it shrinks toward a constant-correlation covariance at a fixed weight of 0.5. The next cell uses only `shrink_cov`. On the latest 60 days and the 100 largest names with complete data, it recomputes the formula and checks it against the function.


## 收缩公式

`shrink_cov` 使用除以 T 的样本协方差 S，目标 F 是平均方差乘以单位阵。`γ = ||S − F||²`，`π` 是每个协方差元素的抽样方差估计之和。收缩强度 `δ = min(π / (γ T), 1)`，估计量是 `(1 − δ) S + δ F`。这是简化的 Ledoit–Wolf，没有原公式里对目标估计误差的 ρ 修正。工具包里的 `shrinkage_cov` 是另一个函数，向常数相关协方差收缩，并且比例固定为 0.5。下面这一格只用 `shrink_cov`。它在最近 60 天、权重最大且数据完整的 100 只股票上，把公式手算一遍，并和函数结果对照。


In [2]:
snapshot = erk.read_hs300_weights().iloc[-2].dropna().sort_values(ascending=False)
window = erk.read_returns().iloc[-60:]
complete = window.columns[window.notna().all()]
names = snapshot.index[snapshot.index.isin(complete)][:100]
window = window[names]

sample = window.cov(ddof=0)
average_variance = float(np.diag(sample).mean())
target = pd.DataFrame(np.eye(len(sample)), index=sample.index, columns=sample.columns) * average_variance
gamma = float(((sample - target) ** 2).to_numpy().sum())
observations = window.shape[0]
centered = window.to_numpy() - window.to_numpy().mean(axis=0)
sample_values = sample.to_numpy()
pi = 0.0
for i in range(sample_values.shape[0]):
    for j in range(sample_values.shape[1]):
        residual = centered[:, i] * centered[:, j] - sample_values[i, j]
        pi += float((residual ** 2).sum())
pi /= observations
delta = min(pi / (gamma * observations), 1.0)
manual = (1 - delta) * sample + delta * target
shrunk = erk.shrink_cov(window)
print(len(names), observations, delta)
print(np.linalg.cond(sample_values), np.linalg.cond(shrunk.to_numpy()))
print(bool(np.allclose(manual.to_numpy(), shrunk.to_numpy())))


100 60 0.08621258178608258
1.0643295793124483e+19 528.2727208362106
True


## Rolling design

The saved results cover 2024-01-02 through 2026-05-29, 580 trading days. Each day uses the index weights from the previous calendar month, drops names with any missing return in the prior 60 days, and keeps the 100 largest weights. The covariance uses the 60 days ending on the previous trading day, so the current day is excluded. Weights come from `weight_gmv`: long only, fully invested, minimizing `w'Σw`, without inverting the covariance. The condition number is stored on that day. Volatility is the standard deviation of portfolio returns over the next 60 trading days, multiplied by `sqrt(252)`. The window moves forward by one day, so neighboring points share 59 days. The two volatility lines are not 580 independent samples.

The calls in the next cell rewrite the two CSVs in `data/`. They stay off unless `regenerate` is set to `True`. The figure uses the run that is already saved.


## 滚动设计

保存下来的结果覆盖 2024-01-02 到 2026-05-29，共 580 个交易日。每一天取上一个自然月的指数权重，丢掉过去 60 天有缺失的股票，留下权重最大的 100 只。协方差用截至前一交易日的 60 天，不包含当天。权重来自 `weight_gmv`：只做多、权重和为 1、最小化 `w'Σw`，不对协方差求逆。条件数记在当天。波动是从当天起 60 个交易日的组合收益标准差，再乘 `sqrt(252)`。窗口每天移动一天，相邻两点重叠 59 天，所以这两条波动不是 580 个独立样本。

下面这个格子里的调用会重写 `data/` 中的两个 CSV。默认不跑；把 `regenerate` 改成 `True` 才会重算。图使用已经保存的那一次结果。


In [3]:
regenerate = False
if regenerate:
    index_weights = erk.read_hs300_weights()
    all_returns = erk.read_returns()
    erk.backtest_hs300(
        all_returns,
        index_weights,
        "2024-01",
        "2026-05",
        weighting=erk.weight_gmv,
        cov_estimator=erk.sample_cov,
    )
    erk.backtest_hs300(
        all_returns,
        index_weights,
        "2024-01",
        "2026-05",
        weighting=erk.weight_gmv,
        cov_estimator=erk.shrink_cov,
    )


## Results

The median condition number of the sample covariance is about `7.5e18`. Sixty trading days have rank at most 59, which cannot span 100 stocks, so the matrix is numerically singular. After `shrink_cov` the median condition number is about 121, and it is lower on every one of the 580 days.

The mean forward realized volatilities are about 11.8% and 12.0%. This minimum-variance problem never inverts `Σ`, so the gap in condition numbers does not become a gap in volatility. Shrinkage turns the matrix into a numerically usable estimate. It does not lower out-of-sample volatility for this long-only minimum-variance portfolio.


## 结果

样本协方差的条件数中位数大约是 `7.5e18`。60 个交易日的秩最多是 59，撑不起 100 只股票，矩阵在数值上不可逆。`shrink_cov` 之后，条件数中位数大约是 121，而且 580 天里每一天都更低。

两条远期已实现波动的均值大约是 11.8% 和 12.0%。这个最小方差没有用到 `Σ` 的逆，所以条件数上的巨大差别没有变成波动差别。收缩把矩阵变成数值上可用的估计；在这个多头最小方差里，样本外波动没有降下来。


In [4]:
sample_result = pd.read_csv("data/drill2_sample_cov_2024-01_2026-05.csv", index_col=0, parse_dates=True)
shrunk_result = pd.read_csv("data/drill2_shrink_cov_2024-01_2026-05.csv", index_col=0, parse_dates=True)
print(len(sample_result), sample_result.index.min().date(), sample_result.index.max().date())
print(float(sample_result["cond"].median()), float(shrunk_result["cond"].median()))
print(float((shrunk_result["cond"] < sample_result["cond"]).mean()))
print(float(sample_result["vol"].mean()), float(shrunk_result["vol"].mean()))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(sample_result.index, np.log10(sample_result["cond"]), label="Sample covariance")
axes[0].plot(shrunk_result.index, np.log10(shrunk_result["cond"]), label="Shrinkage covariance")
axes[0].set_ylabel("log10(condition number)")
axes[0].legend()
axes[1].plot(sample_result.index, sample_result["vol"], label="Sample covariance")
axes[1].plot(shrunk_result.index, shrunk_result["vol"], label="Shrinkage covariance")
axes[1].set_ylabel("Forward 60-day realized volatility")
axes[1].set_xlabel("Date")
axes[1].legend()
fig.tight_layout()
fig.savefig("data/drill2_compare.png", dpi=150)
fig


580 2024-01-02 2026-05-29
7.457553943229701e+18 120.55096490816132
1.0
0.11796186076930588 0.11956148021340772


<Figure size 1000x700 with 2 Axes>